# 05 — Baseline Models

**Joint notebook** — every member runs their model family through the same CV harness so the comparison is fair. Primary metric: PR-AUC (`average_precision`). Recall, precision, F1, balanced accuracy and ROC-AUC are reported beside it.

| Owner | Model |
| --- | --- |
| Meegasthanna | Dummy baseline + Logistic Regression |
| Bandara | XGBoost / LightGBM |
| Seneviratne | Random Forest |
| Umer | SVM (RBF) |

**Input:** the saved train split from notebook 04 (`data/processed/`) and the final 16-feature list saved by notebook 04. Never re-split, never touch the test set here.

**Output:** a shared master results table (baseline, untuned) that notebook 06 extends with tuned results.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import json

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import xgboost as xgb

from src.config import RANDOM_STATE, PROCESSED_DATA_DIR
from src.pipeline import build_preprocessing_pipeline
from src.evaluate import evaluate_cv, METRIC_NAMES

## Load the fixed train split

Load `train.parquet` (saved by notebook 04) and the final feature list. The `cycle` column is the group for `StratifiedGroupKFold`.

In [3]:
train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train.parquet")

with open(PROCESSED_DATA_DIR / "eligible_features.json") as f:
    eligible_features = json.load(f)

binary_cols = ["grip_lost"]
continuous_cols = [c for c in eligible_features if c not in binary_cols]

X_train = train_df[eligible_features]
y_train = train_df["Robot_ProtectiveStop"]
groups = train_df["cycle"]

X_train.shape, y_train.mean()

((5468, 16), 0.03108997805413314)

## CV harness

`evaluate_cv(model, X, y, groups)` lives in `src/evaluate.py` (per the working agreement — shared code lives in `src/`, never copy-pasted between notebooks), so notebook 06 imports the same function instead of redefining it.

Uses `StratifiedGroupKFold` grouped by `cycle` so rows from the same cycle never appear in both the train and validation fold (consistent with the leakage decision in notebook 03). Each model below is passed in as a full `Pipeline` (preprocessing + classifier), not a bare classifier — this way every fold fits its own imputer/scaler on that fold's training data only, instead of reusing a pipeline pre-fit on the whole train set (which would leak validation-fold statistics into the imputer/scaler).

## Master results table

Every model's baseline row gets appended here — mean ± std per metric.

In [4]:
result_columns = ["model", "stage"] + [f"{m}_mean" for m in METRIC_NAMES] + [f"{m}_std" for m in METRIC_NAMES]
results = pd.DataFrame(columns=result_columns)

def add_result(model_name, stage, scores):
    row = {"model": model_name, "stage": stage, **scores}
    results.loc[len(results)] = row

results

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std


## Dummy baseline + Logistic Regression — Meegasthanna

In [5]:
dummy_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
])
dummy_scores = evaluate_cv(dummy_pipeline, X_train, y_train, groups)
add_result("Dummy", "baseline", dummy_scores)

logreg_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE, max_iter=1000)),
])
logreg_scores = evaluate_cv(logreg_pipeline, X_train, y_train, groups)
add_result("Logistic Regression", "baseline", logreg_scores)

results

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std
0,Dummy,baseline,0.031932,0.037326,0.036955,0.036773,0.504153,0.504153,0.005905,0.021369,0.023863,0.022340,0.010333,0.010333
1,Logistic Regression,baseline,0.111114,0.671541,0.065405,0.118519,0.681313,0.736851,0.028067,0.094414,0.014887,0.024752,0.037009,0.068731


## XGBoost / LightGBM — Bandara

In [6]:
xgb_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="aucpr")),
])

xgb_scores = evaluate_cv(xgb_pipeline, X_train, y_train, groups)
add_result("XGBoost", "baseline", xgb_scores)
results

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std
0,Dummy,baseline,0.031932,0.037326,0.036955,0.036773,0.504153,0.504153,0.005905,0.021369,0.023863,0.022340,0.010333,0.010333
1,Logistic Regression,baseline,0.111114,0.671541,0.065405,0.118519,0.681313,0.736851,0.028067,0.094414,0.014887,0.024752,0.037009,0.068731
2,XGBoost,baseline,0.418357,0.307251,0.551728,0.361558,0.648908,0.911487,0.037234,0.133273,0.131464,0.133408,0.064885,0.053311


## Random Forest — Seneviratne

In [7]:
rf_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])

rf_scores = evaluate_cv(rf_pipeline, X_train, y_train, groups)
add_result("Random Forest", "baseline", rf_scores)
results

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std
0,Dummy,baseline,0.031932,0.037326,0.036955,0.036773,0.504153,0.504153,0.005905,0.021369,0.023863,0.022340,0.010333,0.010333
1,Logistic Regression,baseline,0.111114,0.671541,0.065405,0.118519,0.681313,0.736851,0.028067,0.094414,0.014887,0.024752,0.037009,0.068731
2,XGBoost,baseline,0.418357,0.307251,0.551728,0.361558,0.648908,0.911487,0.037234,0.133273,0.131464,0.133408,0.064885,0.053311
3,Random Forest,baseline,0.428161,0.161404,0.478959,0.230981,0.579038,0.914904,0.065739,0.116410,0.254741,0.148579,0.057255,0.055305


## SVM (RBF) — Umer

In [8]:
svm_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE)),
])

svm_scores = evaluate_cv(svm_pipeline, X_train, y_train, groups)
add_result("SVM (RBF)", "baseline", svm_scores)
results

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std
0,Dummy,baseline,0.031932,0.037326,0.036955,0.036773,0.504153,0.504153,0.005905,0.021369,0.023863,0.022340,0.010333,0.010333
1,Logistic Regression,baseline,0.111114,0.671541,0.065405,0.118519,0.681313,0.736851,0.028067,0.094414,0.014887,0.024752,0.037009,0.068731
2,XGBoost,baseline,0.418357,0.307251,0.551728,0.361558,0.648908,0.911487,0.037234,0.133273,0.131464,0.133408,0.064885,0.053311
3,Random Forest,baseline,0.428161,0.161404,0.478959,0.230981,0.579038,0.914904,0.065739,0.116410,0.254741,0.148579,0.057255,0.055305
4,SVM (RBF),baseline,0.254491,0.732430,0.134044,0.223520,0.788816,0.840530,0.038116,0.149087,0.039397,0.058394,0.063932,0.074577


## Results summary

All baselines sorted by PR-AUC. This is the starting point for the 29 Sep results meeting.

In [9]:
results.sort_values("pr_auc_mean", ascending=False)

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std
3,Random Forest,baseline,0.428161,0.161404,0.478959,0.230981,0.579038,0.914904,0.065739,0.116410,0.254741,0.148579,0.057255,0.055305
2,XGBoost,baseline,0.418357,0.307251,0.551728,0.361558,0.648908,0.911487,0.037234,0.133273,0.131464,0.133408,0.064885,0.053311
4,SVM (RBF),baseline,0.254491,0.732430,0.134044,0.223520,0.788816,0.840530,0.038116,0.149087,0.039397,0.058394,0.063932,0.074577
1,Logistic Regression,baseline,0.111114,0.671541,0.065405,0.118519,0.681313,0.736851,0.028067,0.094414,0.014887,0.024752,0.037009,0.068731
0,Dummy,baseline,0.031932,0.037326,0.036955,0.036773,0.504153,0.504153,0.005905,0.021369,0.023863,0.022340,0.010333,0.010333


## Decision log

### Decision: evaluate_cv() fits a fresh preprocessing pipeline per fold, not the notebook-04 pre-fit one
- **Evidence:** The pipeline passed to `evaluate_cv` is a `Pipeline([("preprocessing", build_preprocessing_pipeline(...)), ("classifier", ...)])`, cloned and fit fresh inside each CV fold.
- **Alternative considered:** Reuse the single `preprocessing_pipeline` already fit on the whole train set in notebook 04, and just transform each fold with it.
- **Why rejected:** That would leak validation-fold statistics into the imputer's median and the scaler's IQR (computed across all of train, including rows that fold treats as validation) — a subtler version of the same leakage discipline established in notebook 03. Each fold needs preprocessing fit only on that fold's training rows.

### Decision (Bandara): XGBoost baseline — PR-AUC 0.418, recall 0.307, precision 0.552
- **Evidence:** 5-fold `StratifiedGroupKFold` CV (grouped by cycle). ROC-AUC (0.911) looks strong but is the less meaningful number here given the 3.1% positive rate in the training folds — PR-AUC and recall are what matter for this problem.
- **Alternative considered:** Default `scale_pos_weight=1` vs weighting for the class imbalance.
- **Why rejected (deferred, not rejected):** Left at default for the baseline specifically so tuning (notebook 06) has a clean before/after comparison — `scale_pos_weight` is the first tuning target planned for this model.
- **Why XGBoost suits this data:** Gradient boosting tolerates the correlated features found in notebook 04 (the six temperatures are near-duplicates, |r| ≥ 0.99; the other pairs reach |r| 0.75) without the instability a linear model would have, and it natively supports class-imbalance handling via `scale_pos_weight`, which is why it's the tuning focus.

### Decision (Bandara): Random Forest baseline — PR-AUC 0.428, recall 0.161, precision 0.479
- **Evidence:** Same CV harness and folds as XGBoost. Slightly higher PR-AUC than XGBoost (0.428 vs 0.418) but recall is roughly half (0.161 vs 0.307) — Random Forest is more conservative, missing more true stops in exchange for fewer false alarms.
- **Alternative considered:** `class_weight=None` (default).
- **Why rejected:** `class_weight="balanced"` was used from the baseline onward (unlike XGBoost's deferred `scale_pos_weight`) because Random Forest doesn't have as natural or commonly-tuned an imbalance parameter as `scale_pos_weight` — balanced weighting is the standard starting point for this model.
- **Note for the 29 Sep results meeting:** the recall gap between these two models (0.161 vs 0.307 at the default threshold) is large: a lower-recall model flags fewer of the real stops. How much that matters depends on the operating point (the decision threshold), which is decided in notebooks 06-07, so PR-AUC stays the primary comparison metric.

### Decision (Meegasthanna): Dummy baseline as the theoretical performance floor
- **Evidence:** Stratified Dummy baseline achieves PR-AUC of ~0.031 (matching the positive class prevalence of 3.1%) and F1 of ~0.037 across the 5 CV folds.
- **Alternative considered:** Uniform random or majority-class ("most_frequent") dummy baseline.
- **Why rejected:** "Most_frequent" always predicts 0 (majority class), yielding 0.0 recall, 0.0 precision, and undefined PR-AUC for the minority class. Stratified dummy mirrors the true class distribution and establishes the empirical lower bound that any valid ML model must exceed.

### Decision (Meegasthanna): Logistic Regression baseline — linear benchmark with balanced class weights
- **Evidence:** 5-fold StratifiedGroupKFold CV. `LogisticRegression(class_weight="balanced")` shifts the decision intercept to penalize minority misclassifications, achieving substantial recall but lower precision due to linear separation limits.
- **Alternative considered:** Unweighted `LogisticRegression()` with standard threshold (0.5).
- **Why rejected:** Without `class_weight="balanced"`, unweighted logistic regression collapses toward predicting the majority class (yielding low recall on rare stop events).
- **Why Logistic Regression suits this evaluation:** As the primary linear benchmark, it establishes how much predictive signal is linearly separable before moving to non-linear tree/kernel models (XGBoost, Random Forest, SVM), and serves as an interpretable model for feature coefficient analysis.

### Decision (Umer): SVM (RBF) baseline -- PR-AUC 0.254, recall 0.732, precision 0.134
- **Evidence:** 5-fold `StratifiedGroupKFold` CV (grouped by cycle), same harness and folds as all other models. SVM with RBF kernel and `class_weight="balanced"` provides the baseline for Umer's lane. `probability=True` enables Platt scaling so `predict_proba` returns calibrated probabilities for PR-AUC and ROC-AUC computation, matching the evaluation harness requirements.
- **Alternative considered:** SVM with a linear kernel (`kernel="linear"`), which would be faster and more interpretable.
- **Why rejected:** The RBF kernel can capture non-linear decision boundaries in the sensor feature space -- notebook 02's EDA showed that several sensor families have overlapping distributions between normal and stop classes that a linear hyperplane would struggle to separate. The moderate dimensionality (16 features) keeps the RBF kernel computationally feasible for the ~5,500-row training set.
- **Why `class_weight="balanced"` from the start:** Unlike XGBoost (which deferred `scale_pos_weight` to tuning), SVM's `class_weight="balanced"` is the standard approach for imbalanced data -- it inversely weights the classes by frequency (C_positive ~ 31x C_negative at 3.1% prevalence), preventing the SVM from learning a trivial "always predict majority" boundary. The tuning phase (notebook 06) will explore C and gamma on top of this.
- **Note:** `probability=True` adds a sigmoid calibration step (Platt scaling) after training, fitted with an internal 5-fold cross-validation, so training is several times slower -- acceptable for the baseline but worth revisiting if tuning iterations become slow.
- *Edited by Bandara, 25 Sep:* corrected the fit-time claim (it said "roughly doubles"; scikit-learn fits the sigmoid with an internal 5-fold cross-validation).